# Multi-seed repeated splits — what the bootstrap intervals leave out

`NEXT_STEPS.md` 3.2. Every interval in this project ends with the same caveat:

> *Intervals capture test-set sampling only — split and seed variance are excluded, so they are
> optimistic.*

That sentence has been repeated in `results_summary.md` for the whole project without ever being
measured. This notebook replaces it with a number.

## What is and is not being estimated

The bootstrap in `ablation_significance.ipynb` resamples **the test set**, holding the split and every
model fixed. It answers *"how much would this score move if I had drawn a different 61,235 test rows
from the same distribution?"* — paired band $\pm 0.012$.

It cannot answer the question a reader actually asks: *"if you re-ran the whole thing, would you get
this again?"* Between two runs, **all** of the following change together:

| Source | Changes with the seed? | In the bootstrap? |
|---|---|---|
| which rows land in test | yes | yes |
| which rows land in train | yes | **no** |
| the COMB / SELV partition | yes | **no** |
| the 5 cross-fitting folds | yes | **no** |
| XGBoost column/row subsampling, RF/ET bootstraps | yes | **no** |
| the fitted per-class matrix $W$ | yes | **no** |
| the DEC-space reference set the combiner queries | yes | **no** |

So five sources of variance are excluded and one is included. **Everything is re-run per seed here** —
split, gate, 7-member pool, cross-fit, $W$, reference set, combiner.

## Protocol

Five seeds $\in \{42, 7, 13, 99, 2024\}$. Seed 42 is the published configuration and is included
deliberately: the point is partly to see **whether the reported numbers are typical or lucky**.

The feature spaces are rebuilt identically for every seed — the zero-variance drop, the
$\lvert r\rvert \ge 0.95$ prune and the 31 derived columns are all **label-free**, so they are not part
of what varies (and re-deriving them per seed would confound feature variance with split variance).

Per seed, per space (`BASE` 44, `DERIVED` 75), the full v6 pipeline runs and four combiners are scored
on that seed's test set:

| config | = published model |
|---|---|
| `single a=0.25` | the single-model reference (BASE 0.6157, DERIVED 0.6258) |
| `static W` | **v3** on BASE (0.6334) |
| `dyn k=100 p=8` | **v5** on BASE (0.6363) |
| `dyn k=30 p=8` | **v6** on DERIVED (0.6485) |

**No selection happens here.** Configurations are fixed in advance at the ones already chosen on SELV
in earlier notebooks; a per-seed re-selection would measure something else (and would spend the test
set). Because all configs are scored on the same test rows within a seed, every *contrast* is paired
within seed.

> **Runtime ~2 h** (two cross-fits + two refits per seed). Results are cached to
> `cache_multiseed.csv` after **each seed**, and completed seeds are skipped on re-run, so the
> notebook is restartable.

## 1. Setup — data, the two fixed feature spaces, and the v6 pipeline

In [ ]:
import numpy as np, pandas as pd, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score, precision_score, recall_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

LABELS = {0:"Benign", 1:"Analysis", 2:"Backdoor", 3:"DoS", 4:"Exploits",
          5:"Fuzzers", 6:"Generic", 7:"Reconnaissance", 8:"Shellcode", 9:"Worms"}
NAMES = [LABELS[i] for i in range(10)]
EPS = 1e-6
SEEDS = [42, 7, 13, 99, 2024]
CACHE = Path("cache_multiseed.csv")

Xraw = pd.read_csv("Resource/CIC-UNSW-Data.csv")
yraw = pd.read_csv("Resource/CIC-UNSW-Label.csv")["Label"]
df = pd.concat([Xraw, yraw], axis=1).drop_duplicates().reset_index(drop=True)
Xraw, y = df[Xraw.columns], df["Label"]

nun = Xraw.nunique(); zero = nun[nun <= 1].index.tolist()
FULL = Xraw.drop(columns=zero)
corr = FULL.corr(); upper = corr.abs().where(np.triu(np.ones(corr.shape, bool), k=1))
red = [c for c in upper.columns if (upper[c] >= 0.95).any()]
BASE = FULL.drop(columns=red)

def safe(a, b):
    return (a / (b + EPS)).replace([np.inf, -np.inf], 0.0).fillna(0.0)

# the 31 round-1 derived columns, verbatim from feature_engineering.ipynb / final_model_v6.ipynb
F_ = FULL
g = lambda c: F_[c] if c in F_.columns else pd.Series(0.0, index=F_.index)
fpk, bpk = g("Total Fwd Packet"), g("Total Bwd packets")
fby, bby = g("Total Length of Fwd Packet"), g("Total Length of Bwd Packet")
tot_pk, tot_by = fpk + bpk, fby + bby
d = {}
d["r_pkt_fwd_bwd"] = safe(fpk, bpk); d["r_byte_fwd_bwd"] = safe(fby, bby)
d["asym_pkt"] = safe(fpk - bpk, tot_pk); d["asym_byte"] = safe(fby - bby, tot_by)
d["bpp_fwd"] = safe(fby, fpk); d["bpp_bwd"] = safe(bby, bpk); d["bpp_flow"] = safe(tot_by, tot_pk)
d["r_bpp"] = safe(d["bpp_fwd"], d["bpp_bwd"])
d["cv_flow_iat"] = safe(g("Flow IAT Std"), g("Flow IAT Mean"))
d["cv_fwd_iat"] = safe(g("Fwd IAT Std"), g("Fwd IAT Mean"))
d["cv_bwd_iat"] = safe(g("Bwd IAT Std"), g("Bwd IAT Mean"))
d["iat_span_rel"] = safe(g("Flow IAT Max") - g("Flow IAT Min"), g("Flow IAT Mean"))
d["cv_fwd_len"] = safe(g("Fwd Packet Length Std"), g("Fwd Packet Length Mean"))
d["cv_bwd_len"] = safe(g("Bwd Packet Length Std"), g("Bwd Packet Length Mean"))
d["cv_pkt_len"] = safe(g("Packet Length Std"), g("Packet Length Mean"))
d["fwd_len_span_rel"] = safe(g("Fwd Packet Length Max") - g("Fwd Packet Length Min"), g("Fwd Packet Length Mean"))
d["bwd_len_span_rel"] = safe(g("Bwd Packet Length Max") - g("Bwd Packet Length Min"), g("Bwd Packet Length Mean"))
for flag in ("FIN", "SYN", "RST", "PSH", "ACK"):
    d["frac_" + flag] = safe(g(flag + " Flag Count"), tot_pk)
d["r_syn_fin"] = safe(g("SYN Flag Count"), g("FIN Flag Count"))
d["hdr_per_pkt_fwd"] = safe(g("Fwd Header Length"), fpk)
d["hdr_per_pkt_bwd"] = safe(g("Bwd Header Length"), bpk)
d["payload_share_fwd"] = safe(fby - g("Fwd Header Length"), fby)
dur = g("Flow Duration")
d["active_share"] = safe(g("Active Mean"), dur); d["idle_share"] = safe(g("Idle Mean"), dur)
d["r_active_idle"] = safe(g("Active Mean"), g("Idle Mean")); d["dur_per_pkt"] = safe(dur, tot_pk)
d["r_init_win"] = safe(g("FWD Init Win Bytes"), g("Bwd Init Win Bytes"))
DER = pd.DataFrame(d, index=F_.index).replace([np.inf, -np.inf], 0.0).fillna(0.0)
dz = DER.nunique(); DER = DER.drop(columns=dz[dz <= 1].index.tolist())
DERIVED = pd.concat([BASE, DER], axis=1)

SPACES = {"BASE": BASE, "DERIVED": DERIVED}
attack_labels = sorted(l for l in y.unique() if l != 0)
fam_to_idx = {f: i for i, f in enumerate(attack_labels)}
idx_to_fam = {i: f for f, i in fam_to_idx.items()}
NC = len(attack_labels); FAM = [LABELS[idx_to_fam[i]] for i in range(NC)]
print("rows %d | BASE %d feat | DERIVED %d feat | seeds %s"
      % (len(df), BASE.shape[1], DERIVED.shape[1], SEEDS))

In [ ]:
def make_xgb(seed, **kw):
    p = dict(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.9,
             colsample_bytree=0.9, tree_method="hist", eval_metric="mlogloss",
             n_jobs=-1, random_state=seed)
    p.update(kw); return XGBClassifier(**p)

def wc_alpha(n, a):
    w = (n.sum() / n) ** a; return w / w.mean()

def sample_w(ya, wc):
    w = wc[ya].astype(float); return w * len(w) / w.sum()

SPECS = ([(f"XGB a={a}", ("alpha", a)) for a in (0.0, 0.25, 0.5, 1.0)]
         + [("XGB smote2000", ("smote", 2000)),
            ("RF a=0.25", ("rf", 0.25)), ("ET a=0.25", ("et", 0.25))])
MODELS = [s[0] for s in SPECS]; M = len(MODELS)
I25 = MODELS.index("XGB a=0.25")

def fit_member(kind, param, Xa, ya, seed):
    n = np.bincount(np.asarray(ya), minlength=NC)
    if kind == "smote":
        strat = {c: param for c in range(NC) if n[c] < param}
        Xs, ys = SMOTE(random_state=seed, sampling_strategy=strat).fit_resample(Xa, ya)
        m = make_xgb(seed); m.fit(Xs, ys); return m
    if   kind == "alpha": m = make_xgb(seed)
    elif kind == "rf":    m = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=seed)
    elif kind == "et":    m = ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=seed)
    m.fit(Xa, ya, sample_weight=sample_w(np.asarray(ya), wc_alpha(n, param)))
    return m

def fit_gate(Xa, ya10, seed):
    yb = (ya10 != 0).astype(int)
    m = make_xgb(seed); m.fit(Xa, yb, sample_weight=compute_sample_weight("balanced", yb)); return m

def fuse_static(probas, Wclass):
    S = np.zeros(probas[0].shape, np.float64)
    for m in range(M): S += probas[m] * Wclass[m][None, :]
    return S.argmax(1)

def fuse_dyn(probas, Wd, p=1.0, Wclass=None):
    Wp = np.power(np.clip(Wd, 1e-6, None), p); Wp = Wp / Wp.sum(1, keepdims=True)
    S = np.zeros(probas[0].shape, np.float64)
    for m in range(M):
        coef = Wp[:, m:m+1] if Wclass is None else Wp[:, m:m+1] * Wclass[m][None, :]
        S += coef * probas[m]
    return S.argmax(1)

GRID_W = (0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0)
def fit_W(probas, ytrue, passes=3):
    W = np.ones((M, NC)) / M
    best = f1_score(ytrue, fuse_static(probas, W), average="macro", zero_division=0)
    for _ in range(passes):
        for c in range(NC):
            for m_ in range(M):
                cur = W[m_, c]
                for gg in GRID_W:
                    W[m_, c] = gg
                    if W[:, c].sum() == 0: continue
                    s = f1_score(ytrue, fuse_static(probas, W), average="macro", zero_division=0)
                    if s > best + 1e-7: best, cur = s, gg
                W[m_, c] = cur
    return W

prof = lambda ps: np.hstack(ps).astype(np.float32)
print("pool:", M, "members |", ", ".join(MODELS))

### 1.1 One seed, one space — the whole v6 pipeline

Identical in every respect to `final_model_v6.ipynb` except that the split seed, the fold seed, the
COMB/SELV seed and every model's `random_state` are driven by the loop variable instead of being
fixed at 42.

In [ ]:
def run(space_name, X, seed, verbose=True):
    t_all = time.time()
    X_trf, X_test, y_trf, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=seed, stratify=y)
    X_trf = X_trf.reset_index(drop=True); y_trf = y_trf.reset_index(drop=True)
    y_arr = y_trf.values; att = y_arr != 0
    fam = np.full(len(y_arr), -1)
    fam[att] = pd.Series(y_arr[att]).map(fam_to_idx).values

    # --- 5-fold cross-fit for OOF probabilities (the combiner's fitting data) ---
    OOF = [np.zeros((len(y_arr), NC), np.float32) for _ in range(M)]
    oof_gate = np.zeros(len(y_arr), int)
    t0 = time.time()
    for tr, va in StratifiedKFold(5, shuffle=True, random_state=seed).split(X_trf, y_arr):
        Xtr_f, ytr_f = X_trf.iloc[tr], y_arr[tr]
        oof_gate[va] = fit_gate(Xtr_f, ytr_f, seed).predict(X_trf.iloc[va])
        ma = ytr_f != 0
        Xa = Xtr_f[ma].reset_index(drop=True); ya = pd.Series(fam[tr][ma])
        for mi, (_, (k, p)) in enumerate(SPECS):
            OOF[mi][va] = fit_member(k, p, Xa, ya, seed).predict_proba(X_trf.iloc[va]).astype(np.float32)
    t_cf = time.time() - t0

    # --- combiner: W on COMB half, DEC-space reference set from the same rows ---
    comb_idx, _ = train_test_split(np.arange(len(y_arr)), test_size=0.5,
                                   random_state=seed + 1, stratify=y_arr)
    ref_idx = comb_idx[att[comb_idx]]
    y_ref = fam[ref_idx]
    CORRECT = np.stack([(OOF[m][ref_idx].argmax(1) == y_ref) for m in range(M)])
    WC_REF = wc_alpha(np.bincount(y_ref, minlength=NC), 0.25)
    W_class = fit_W([OOF[m][ref_idx] for m in range(M)], y_ref)
    NN_D = NearestNeighbors(n_neighbors=100, algorithm="brute", n_jobs=-1).fit(
        prof([OOF[m][ref_idx] for m in range(M)]))

    # --- refit pool + gate on all of train+val, then touch test once ---
    t0 = time.time()
    gate_f = fit_gate(X_trf, y_arr, seed)
    Xa_f = X_trf[att].reset_index(drop=True); ya_f = pd.Series(fam[att])
    FINAL = [fit_member(k, p, Xa_f, ya_f, seed) for _, (k, p) in SPECS]
    t_rf = time.time() - t0

    yte = y_test.values
    gmask = gate_f.predict(X_test) == 1
    gpos = np.where(gmask)[0]
    P = [m.predict_proba(X_test.iloc[gpos]).astype(np.float32) for m in FINAL]
    _, idx = NN_D.kneighbors(prof(P))

    def comp_ola(k):
        nb = idx[:, :k]; wn = WC_REF[y_ref[nb]]; den = wn.sum(1)
        Wd = np.empty((nb.shape[0], M))
        for m in range(M):
            Wd[:, m] = (CORRECT[m][nb] * wn).sum(1) / den
        return Wd

    variants = {
        "single a=0.25":  P[I25].argmax(1),
        "static W":       fuse_static(P, W_class),
        "dyn k=100 p=8":  fuse_dyn(P, comp_ola(100), p=8),
        "dyn k=30 p=8":   fuse_dyn(P, comp_ola(30), p=8),
    }
    rows = []
    for tag, fam_g in variants.items():
        yp = np.zeros(len(yte), int)
        yp[gmask] = np.vectorize(idx_to_fam.get, otypes=[int])(fam_g)
        r = dict(seed=seed, space=space_name, config=tag,
                 mF1=f1_score(yte, yp, average="macro", zero_division=0),
                 mP=precision_score(yte, yp, average="macro", zero_division=0),
                 mR=recall_score(yte, yp, average="macro", zero_division=0),
                 acc=float((yp == yte).mean()),
                 gate_P=precision_score((yte != 0).astype(int), gmask.astype(int)),
                 gate_R=recall_score((yte != 0).astype(int), gmask.astype(int)))
        for ci, f1c in enumerate(f1_score(yte, yp, average=None, labels=range(10), zero_division=0)):
            r["f1_" + NAMES[ci]] = f1c
        rows.append(r)
    if verbose:
        print("    %-8s seed %-5d crossfit %3.0fs refit %3.0fs total %4.0fs | mF1 "
              % (space_name, seed, t_cf, t_rf, time.time() - t_all)
              + " ".join("%s %.4f" % (t.split()[0], r["mF1"]) for t, r in zip(variants, rows)))
    return rows

## 2. Run the grid — 5 seeds x 2 spaces

Cached after every seed; re-running skips what is already in `cache_multiseed.csv`.

In [ ]:
done = pd.read_csv(CACHE) if CACHE.exists() else pd.DataFrame()
have = set(zip(done.seed, done.space)) if len(done) else set()
allrows = done.to_dict("records") if len(done) else []

t_start = time.time()
for seed in SEEDS:
    for sname, Xs in SPACES.items():
        if (seed, sname) in have:
            print("  skip %s seed %d (cached)" % (sname, seed)); continue
        allrows += run(sname, Xs, seed)
        pd.DataFrame(allrows).to_csv(CACHE, index=False)
print("\ngrid complete in %.0f min" % ((time.time() - t_start) / 60))

R = pd.DataFrame(allrows)
R["model"] = R.space + " / " + R.config
print(R.pivot_table(index="model", columns="seed", values="mF1").round(4).to_string())

## 3. How much does the score move between runs?

`sd_seed` is the across-seed standard deviation of the **whole pipeline** — the quantity every
interval in this project has so far omitted. Compare it against the bootstrap's single-configuration
half-width of $\pm 0.017$, which came from test-set resampling alone.

In [ ]:
KEY = [("BASE", "single a=0.25", 0.6157), ("BASE", "static W", 0.6334),
       ("BASE", "dyn k=100 p=8", 0.6363), ("DERIVED", "single a=0.25", 0.6258),
       ("DERIVED", "static W", None), ("DERIVED", "dyn k=30 p=8", 0.6485)]
rows = []
for sp, cf, pub in KEY:
    g_ = R[(R.space == sp) & (R.config == cf)]
    if not len(g_): continue
    v = g_.sort_values("seed").mF1.values
    a = g_.sort_values("seed").acc.values
    s42 = g_[g_.seed == 42].mF1
    rows.append(dict(space=sp, config=cf, published=pub,
                     seed42=round(float(s42.iloc[0]), 4) if len(s42) else np.nan,
                     mean=round(v.mean(), 4), sd_seed=round(v.std(ddof=1), 4),
                     min=round(v.min(), 4), max=round(v.max(), 4),
                     spread=round(v.max() - v.min(), 4),
                     acc_mean=round(a.mean(), 4), acc_sd=round(a.std(ddof=1), 4)))
S = pd.DataFrame(rows)
print(S.to_string(index=False))
print("\nbootstrap half-width for a single configuration was +/-0.017 (test resampling only).")
print("across-seed 95%% interval for the final model, mean +/- t(4)*sd/sqrt(5):")
f = S[(S.space == "DERIVED") & (S.config == "dyn k=30 p=8")].iloc[0]
hw = 2.776 * f["sd_seed"] / np.sqrt(5)
print("  v6 macro-F1 %.4f +/- %.4f  ->  [%.4f, %.4f]   (sd across seeds %.4f)"
      % (f["mean"], hw, f["mean"] - hw, f["mean"] + hw, f["sd_seed"]))
print("  a single run's spread across the five seeds was %.4f (min %.4f, max %.4f)"
      % (f["spread"], f["min"], f["max"]))

### 3.1 Was seed 42 lucky?

Every headline number in the project comes from seed 42. If it sits near the mean of its own
distribution the published figures are representative; if it sits at an extreme they are not.

In [ ]:
rows = []
for sp, cf, pub in KEY:
    g_ = R[(R.space == sp) & (R.config == cf)]
    if not len(g_) or 42 not in set(g_.seed): continue
    v = g_.mF1.values; v42 = float(g_[g_.seed == 42].mF1.iloc[0])
    sd = v.std(ddof=1)
    rows.append(dict(space=sp, config=cf, seed42=round(v42, 4),
                     mean_other=round(v[g_.seed.values != 42].mean(), 4),
                     z=round((v42 - v.mean()) / sd, 2) if sd > 0 else np.nan,
                     rank_of_5=int((v > v42).sum() + 1)))
print(pd.DataFrame(rows).to_string(index=False))
print("\nz = how many across-seed SDs seed 42 sits above its own mean; rank 1 = best of the five.")

## 4. Do the established contrasts survive re-splitting?

The comparisons in `v6_significance.ipynb` were paired **within** the seed-42 test set. Here each
contrast is recomputed within every seed (still paired — same test rows, same run) and the variation
of the *difference* across seeds is what is reported.

A contrast survives if its across-seed interval excludes zero. With only 5 seeds this is a
low-powered test, so a contrast that fails here is **not** thereby refuted — but one that survives is
considerably better evidenced than the bootstrap alone can show.

In [ ]:
def get(sp, cf, seed, col="mF1"):
    g_ = R[(R.space == sp) & (R.config == cf) & (R.seed == seed)]
    return float(g_[col].iloc[0]) if len(g_) else np.nan

CONTRASTS = [
    ("v6 - v5            (DERIVED dyn30 - BASE dyn100)", ("DERIVED","dyn k=30 p=8"), ("BASE","dyn k=100 p=8"), 0.0123),
    ("v6 - v3            (DERIVED dyn30 - BASE staticW)", ("DERIVED","dyn k=30 p=8"), ("BASE","static W"), 0.0152),
    ("v6 - single DERIVED (ensembling, on DERIVED)",      ("DERIVED","dyn k=30 p=8"), ("DERIVED","single a=0.25"), 0.0231),
    ("single DERIVED - single BASE (features alone)",     ("DERIVED","single a=0.25"), ("BASE","single a=0.25"), 0.0100),
    ("v5 - v3            (dynamic - static, on BASE)",    ("BASE","dyn k=100 p=8"), ("BASE","static W"), 0.0029),
    ("dyn30 - staticW    (dynamic - static, on DERIVED)", ("DERIVED","dyn k=30 p=8"), ("DERIVED","static W"), None),
]
rows = []
for tag, A, B, boot in CONTRASTS:
    dd = np.array([get(*A, s) - get(*B, s) for s in SEEDS])
    dd = dd[~np.isnan(dd)]
    if len(dd) < 2: continue
    sd = dd.std(ddof=1); hw = 2.776 * sd / np.sqrt(len(dd))
    rows.append(dict(contrast=tag, boot_seed42=boot, mean=round(dd.mean(), 4),
                     sd=round(sd, 4), lo=round(dd.mean() - hw, 4), hi=round(dd.mean() + hw, 4),
                     pos_of_5=int((dd > 0).sum()),
                     survives=("YES" if dd.mean() - hw > 0 else
                               ("negative" if dd.mean() + hw < 0 else "no"))))
Cn = pd.DataFrame(rows)
print(Cn.to_string(index=False))
print("\nboot_seed42 = the paired-bootstrap point estimate from v6_significance.ipynb (seed 42 only).")
print("pos_of_5    = in how many of the five independent runs the difference had the expected sign.")

### 4.1 The two variance components side by side

For the final model, how does re-splitting compare with resampling the test set? If `sd_seed` is
appreciably larger than the bootstrap SD, then every interval in `results_summary.md` is too narrow
and the correction belongs in the thesis.

In [ ]:
boot_sd_single = 0.0350 / (2 * 1.96)   # mean 95% CI width 0.0350 -> SD, ablation_significance.ipynb
boot_sd_paired = 0.0240 / (2 * 1.96)   # paired band +/-0.012
f = S[(S.space == "DERIVED") & (S.config == "dyn k=30 p=8")].iloc[0]
print("final model (v6) macro-F1")
print("  SD from test-set resampling, split fixed  : %.4f   (bootstrap, ablation_significance)" % boot_sd_single)
print("  SD from re-running the whole pipeline     : %.4f   (this notebook, 5 seeds)" % f["sd_seed"])
print("  ratio                                     : %.2fx" % (f["sd_seed"] / boot_sd_single))
print()
for tag, A, B, boot in CONTRASTS[:3]:
    dd = np.array([get(*A, s) - get(*B, s) for s in SEEDS]); dd = dd[~np.isnan(dd)]
    print("  contrast SD across seeds  %-52s %.4f  (bootstrap paired SD %.4f)"
          % (tag.split("(")[0].strip(), dd.std(ddof=1), boot_sd_paired))

## 5. Rare classes — the Worms caveat, measured

`v6_significance.ipynb` flagged that Worms (48 test rows) carries 57% of v6's headline gain and that
excluding it the gain is not established. The bootstrap gave Worms a $\pm 0.13$ interval from test
resampling alone. Across seeds the class membership of the test set changes too, so this is the
stricter measurement.

In [ ]:
f6 = R[(R.space == "DERIVED") & (R.config == "dyn k=30 p=8")].sort_values("seed")
boot = {"Benign":0.002,"Fuzzers":0.013,"Exploits":0.018,"Reconnaissance":0.029,"Generic":0.038,
        "DoS":0.061,"Shellcode":0.088,"Analysis":0.132,"Backdoor":0.184,"Worms":0.260}
rows = []
for n in NAMES:
    v = f6["f1_" + n].values
    rows.append(dict(cls=n, mean=round(v.mean(), 3), sd_seed=round(v.std(ddof=1), 4),
                     min=round(v.min(), 3), max=round(v.max(), 3),
                     spread=round(v.max() - v.min(), 3),
                     boot_width=boot[n], seed_width=round(2 * 2.776 * v.std(ddof=1) / np.sqrt(len(v)), 3)))
PC = pd.DataFrame(rows)
print("per-class F1 of the final model across %d seeds" % len(f6))
print(PC.to_string(index=False))
print("\nboot_width  = 95%% CI width from test resampling at seed 42 (ablation_significance.ipynb)")
print("seed_width  = 95%% CI width of the mean across seeds (t(4))")

print("\nmacro-F1 excluding Worms, per seed:")
cols = ["f1_" + n for n in NAMES if n != "Worms"]
for _, r in f6.iterrows():
    print("  seed %-5d full %.4f | ex-Worms %.4f" % (r["seed"], r["mF1"], r[cols].mean()))
exw = np.array([r[cols].mean() for _, r in f6.iterrows()])
print("  ex-Worms across seeds: mean %.4f sd %.4f" % (exw.mean(), exw.std(ddof=1)))

## 6. Findings

_Filled in from the outputs above once the notebook has run._